# TakeMeter — Fine-Tuning Notebook

**Before running anything:** go to `Runtime` → `Change runtime type` → select **T4 GPU** → Save.

Run the sections in order:
1. Setup + upload your labeled CSV
2. Split + tokenize
3. Fine-tune DistilBERT
4. Evaluate fine-tuned model
5. Groq zero-shot baseline
6. Compare both + export results

⚠️ Section 5 (baseline) needs Sections 1 and 2 to have run first, even though it doesn't use the fine-tuned model.

## Section 1 — Setup, label map, upload CSV

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate groq

import pandas as pd
import numpy as np
import torch
import json
from google.colab import files

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠ No GPU detected — go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Label map — must match your planning.md taxonomy exactly
LABEL2ID = {"analysis": 0, "hot_take": 1, "question": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print(LABEL2ID)

In [ ]:
# Upload your labeled CSV (data/raw_posts.csv from your local project)
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
print(f"Uploaded: {csv_filename}")

In [ ]:
df = pd.read_csv(csv_filename)

# Keep only labeled rows with valid labels
df = df[df["label"].isin(LABEL2ID.keys())].copy()
df["label_id"] = df["label"].map(LABEL2ID)
df = df[["text", "label", "label_id"]].reset_index(drop=True)

print(f"Total labeled rows: {len(df)}")
print("\nLabel distribution:")
print(df["label"].value_counts())

if len(df) < 100:
    print("\n⚠ Warning: fewer than 100 labeled rows. Check your CSV upload.")

## Section 2 — Split + tokenize

In [ ]:
from sklearn.model_selection import train_test_split

# 70% train / 15% val / 15% test, stratified by label
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"]
)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print("\nTrain distribution:\n", train_df["label"].value_counts())
print("\nVal distribution:\n", val_df["label"].value_counts())
print("\nTest distribution:\n", test_df["label"].value_counts())

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(d):
    return Dataset.from_pandas(d[["text", "label_id"]].rename(columns={"label_id": "label"}).reset_index(drop=True))

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print("Tokenization complete.")
print(train_ds[0].keys())

## Section 3 — Fine-tune DistilBERT

Default hyperparameters: 3 epochs, learning rate 2e-5, batch size 16.
Training takes roughly 5–15 minutes on a T4 GPU for ~250 examples.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABEL2ID), id2label=ID2LABEL, label2id=LABEL2ID
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "macro_precision": precision, "macro_recall": recall, "macro_f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,            # ← hyperparameter: epochs
    learning_rate=2e-5,            # ← hyperparameter: learning rate
    per_device_train_batch_size=16,# ← hyperparameter: batch size
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

## Section 4 — Evaluate fine-tuned model on test set

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

test_predictions = trainer.predict(test_ds)
test_preds = np.argmax(test_predictions.predictions, axis=-1)
test_probs = torch.softmax(torch.tensor(test_predictions.predictions), dim=-1).numpy()
test_labels = test_predictions.label_ids

ft_accuracy = accuracy_score(test_labels, test_preds)
print(f"Fine-tuned model test accuracy: {ft_accuracy:.4f}\n")

print("Per-class report:")
report_dict = classification_report(
    test_labels, test_preds,
    target_names=[ID2LABEL[i] for i in range(len(LABEL2ID))],
    output_dict=True, zero_division=0
)
print(classification_report(
    test_labels, test_preds,
    target_names=[ID2LABEL[i] for i in range(len(LABEL2ID))],
    zero_division=0
))

cm = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[ID2LABEL[i] for i in range(len(LABEL2ID))])
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Fine-tuned DistilBERT — Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print("\n✓ Saved confusion_matrix.png")

In [ ]:
# Look at wrong predictions — pick 3+ for your README analysis
wrong_mask = test_preds != test_labels
wrong_df = test_df.reset_index(drop=True)[wrong_mask].copy()
wrong_df["predicted"] = [ID2LABEL[p] for p in test_preds[wrong_mask]]
wrong_df["confidence"] = test_probs[wrong_mask].max(axis=1)

print(f"Total wrong predictions: {len(wrong_df)} out of {len(test_df)}\n")
for idx, row in wrong_df.head(10).iterrows():
    print(f"True: {row['label']}  |  Predicted: {row['predicted']}  |  Confidence: {row['confidence']:.2f}")
    print(f"Text: {row['text'][:200]}")
    print("-" * 80)

## Section 5 — Groq zero-shot baseline

Get a free API key at https://console.groq.com if you don't have one.
Add it via Colab Secrets (🔑 icon, left sidebar) named `GROQ_API_KEY`, or paste directly below (don't commit it to GitHub).

In [ ]:
from groq import Groq
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = input("Paste your Groq API key: ").strip()

groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready.")

In [ ]:
# Classification prompt — uses the exact label definitions from planning.md
SYSTEM_PROMPT = """You are a discourse classifier for AI/tech community posts. \
Classify each post into EXACTLY ONE of these three labels:

analysis: The post makes a structured argument supported by specific, verifiable evidence \
(benchmarks, paper citations, documented results, technical reasoning). The claim could be \
checked or falsified using the evidence provided.

hot_take: A bold, confident opinion stated WITHOUT substantive supporting evidence. The post \
asserts rather than argues. Framing is often provocative.

question: The post primarily asks for information, explanation, recommendations, or help. \
The author's goal is to receive knowledge from others, not argue a position.

Decision rule for analysis vs hot_take: if the evidence would convince a skeptic even after \
removing opinion language, label analysis. If the evidence is decorative (a name-drop, vague \
reference) while the core claim is asserted without reasoning, label hot_take.

Respond with ONLY the label name — exactly one of: analysis, hot_take, question
No explanation, no punctuation, no other text."""

def classify_with_groq(text):
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text}
            ],
            temperature=0,
            max_tokens=10,
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        return f"ERROR: {e}"

# Quick test
test_result = classify_with_groq(test_df.iloc[0]["text"])
print(f"Test classification: {test_result}")
print(f"Actual label: {test_df.iloc[0]['label']}")

In [ ]:
import time

baseline_preds = []
unparseable = 0

for i, row in test_df.reset_index(drop=True).iterrows():
    pred = classify_with_groq(row["text"])
    # Clean up common formatting issues
    pred_clean = pred.strip().lower().replace(".", "").replace("'", "").replace('"', "")
    
    if pred_clean not in LABEL2ID:
        unparseable += 1
        # try to salvage by checking substring match
        matched = None
        for label in LABEL2ID:
            if label in pred_clean:
                matched = label
                break
        pred_clean = matched if matched else "question"  # fallback
    
    baseline_preds.append(pred_clean)
    
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(test_df)} classified...")
    time.sleep(0.3)  # be polite to rate limits

print(f"\nDone. {unparseable} unparseable responses out of {len(test_df)} " 
      f"({unparseable/len(test_df)*100:.1f}%)")
if unparseable / len(test_df) > 0.10:
    print("⚠ More than 10% unparseable — consider revising the prompt.")

In [ ]:
baseline_pred_ids = [LABEL2ID[p] for p in baseline_preds]
baseline_true_ids = test_df["label_id"].values

baseline_accuracy = accuracy_score(baseline_true_ids, baseline_pred_ids)
print(f"Groq zero-shot baseline accuracy: {baseline_accuracy:.4f}\n")

print("Per-class report (baseline):")
baseline_report_dict = classification_report(
    baseline_true_ids, baseline_pred_ids,
    target_names=[ID2LABEL[i] for i in range(len(LABEL2ID))],
    output_dict=True, zero_division=0
)
print(classification_report(
    baseline_true_ids, baseline_pred_ids,
    target_names=[ID2LABEL[i] for i in range(len(LABEL2ID))],
    zero_division=0
))

## Section 6 — Compare both models + export results

In [ ]:
print("=" * 60)
print("BASELINE vs FINE-TUNED COMPARISON")
print("=" * 60)
print(f"{'Metric':<20}{'Baseline (Groq)':<20}{'Fine-tuned':<20}")
print("-" * 60)
print(f"{'Accuracy':<20}{baseline_accuracy:<20.4f}{ft_accuracy:<20.4f}")
print(f"{'Macro F1':<20}{baseline_report_dict['macro avg']['f1-score']:<20.4f}{report_dict['macro avg']['f1-score']:<20.4f}")
print("=" * 60)

improvement = (ft_accuracy - baseline_accuracy) * 100
print(f"\nFine-tuning improved accuracy by {improvement:+.1f} percentage points")

In [ ]:
# Sample classifications table for your README
sample_df = test_df.reset_index(drop=True).sample(min(5, len(test_df)), random_state=1).copy()
sample_indices = sample_df.index

print("Sample classifications (fine-tuned model):\n")
for idx in sample_indices:
    row = test_df.reset_index(drop=True).iloc[idx]
    pred_label = ID2LABEL[test_preds[idx]]
    conf = test_probs[idx].max()
    correct = "✓" if pred_label == row["label"] else "✗"
    print(f"{correct} True: {row['label']:<10} Predicted: {pred_label:<10} Confidence: {conf:.2f}")
    print(f"  {row['text'][:150]}")
    print()

In [ ]:
# Export everything to JSON for your README / evaluation report
results = {
    "baseline": {
        "model": "llama-3.3-70b-versatile (Groq, zero-shot)",
        "accuracy": float(baseline_accuracy),
        "macro_f1": float(baseline_report_dict["macro avg"]["f1-score"]),
        "per_class": {
            label: {
                "precision": float(baseline_report_dict[label]["precision"]),
                "recall": float(baseline_report_dict[label]["recall"]),
                "f1": float(baseline_report_dict[label]["f1-score"]),
            }
            for label in LABEL2ID
        },
        "unparseable_pct": float(unparseable / len(test_df) * 100),
    },
    "fine_tuned": {
        "model": "distilbert-base-uncased (fine-tuned)",
        "hyperparameters": {
            "epochs": 3,
            "learning_rate": 2e-5,
            "batch_size": 16,
        },
        "accuracy": float(ft_accuracy),
        "macro_f1": float(report_dict["macro avg"]["f1-score"]),
        "per_class": {
            label: {
                "precision": float(report_dict[label]["precision"]),
                "recall": float(report_dict[label]["recall"]),
                "f1": float(report_dict[label]["f1-score"]),
            }
            for label in LABEL2ID
        },
    },
    "confusion_matrix": cm.tolist(),
    "confusion_matrix_labels": [ID2LABEL[i] for i in range(len(LABEL2ID))],
    "test_set_size": len(test_df),
    "wrong_predictions_sample": [
        {
            "text": row["text"],
            "true_label": row["label"],
            "predicted": row["predicted"],
            "confidence": float(row["confidence"]),
        }
        for _, row in wrong_df.head(5).iterrows()
    ],
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("✓ Saved evaluation_results.json")
print(json.dumps(results, indent=2)[:1000] + "...")

In [ ]:
# Download both output files to your computer
files.download("evaluation_results.json")
files.download("confusion_matrix.png")
print("Files downloaded. Move them into your project's outputs/ folder.")